In [1]:
!pip -q install timm gdown scikit-learn

In [2]:
from google.colab import drive
import os

drive.mount('/content/drive')

ROOT = "/content/drive/MyDrive/dataset/MILK10k_Split"

MODEL_NAME = "ghostnetv2"
WIDTH      = 1.0
DROPOUT    = 0.2
PRETRAINED = True
IMG_SIZE   = 224
BATCH_SIZE = 16
EPOCHS     = 100
LR         = 0.0001
WEIGHT_DECAY = 0.05
PATIENCE   = 5
SEED       = 42
NUM_WORKERS = 4

assert os.path.exists(ROOT), f"Dataset folder not found: {ROOT}"

for s in ["train", "val", "test"]:
    split_path = os.path.join(ROOT, s)
    assert os.path.exists(split_path), f"{s} folder not found: {split_path}"
    print(s, "->", os.listdir(split_path))

print("\nDataset root:", ROOT)
print("Dataset found successfully.")

Mounted at /content/drive
train -> ['GroundTruth.csv', 'Metadata.csv', 'Supplement_modified train.csv', 'images']
val -> ['GroundTruth.csv', 'Metadata.csv', 'Supplement_modified val.csv', 'images']
test -> ['GroundTruth.csv', 'Metadata.csv', 'Supplement_modified.csv', 'images']

Dataset root: /content/drive/MyDrive/dataset/MILK10k_Split
Dataset found successfully.


In [4]:
import pandas as pd, numpy as np, torch, random,glob

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "(!! turn on GPU runtime)")

CLASSES = ["melanoma", "nevus", "bcc", "scc", "keratoses"]
CLASS2IDX = {c: i for i, c in enumerate(CLASSES)}

def map_label(name):
    s = str(name).lower()
    if "collision" in s:            return None
    if "melanoma" in s:             return "melanoma"
    if "squamous cell carcinoma" in s: return "scc"
    if "basal cell carcinoma" in s: return "bcc"
    if "keratosis" in s:            return "keratoses"
    if "nevus" in s:                return "nevus"
    return None

def load_split(split):
    split_dir = os.path.join(ROOT, split)
    csv_candidates = glob.glob(os.path.join(split_dir, "*Supplement_modified*"))
    if not csv_candidates:
        csv_candidates = glob.glob(os.path.join(split_dir, "*.csv"))

    csv = csv_candidates[0]
    print(f"Loading {split} from: {os.path.basename(csv)}")
    df = pd.read_csv(csv)
    df["cls"] = df["diagnosis_full"].map(map_label)
    n_all = len(df)
    df = df.dropna(subset=["cls"]).copy()
    df["path"] = df["isic_id"].apply(lambda i: os.path.join(split_dir, "images", f"{i}.jpg"))
    df = df[df["path"].apply(os.path.exists)].reset_index(drop=True)
    df["label"] = df["cls"].map(CLASS2IDX)
    print(f"{split:5s}: kept {len(df)} / {n_all} images | {df['cls'].value_counts().to_dict()}")
    return df

train_df, val_df, test_df = load_split("train"), load_split("val"), load_split("test")

Device: cuda Tesla T4
Loading train from: Supplement_modified train.csv
train: kept 7686 / 8384 images | {'bcc': 4052, 'nevus': 1154, 'keratoses': 948, 'scc': 802, 'melanoma': 730}
Loading val from: Supplement_modified val.csv
val  : kept 970 / 1048 images | {'bcc': 510, 'nevus': 162, 'keratoses': 134, 'melanoma': 86, 'scc': 78}
Loading test from: Supplement_modified.csv
test : kept 964 / 1048 images | {'bcc': 482, 'nevus': 176, 'keratoses': 120, 'scc': 102, 'melanoma': 84}


In [5]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(20),
    T.ColorJitter(0.2, 0.2, 0.2, 0.02),
    T.ToTensor(), T.Normalize(MEAN, STD)])
eval_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)])

class LesionDS(Dataset):
    def __init__(self, df, tf): self.df, self.tf = df.reset_index(drop=True), tf
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        return self.tf(Image.open(r["path"]).convert("RGB")), int(r["label"])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(LesionDS(val_df,   eval_tf),  BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(LesionDS(test_df,  eval_tf),  BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [6]:
import math
import torch.nn as nn

# ============ GhostNetV2 architecture (copied unchanged from ghostnetv2_combined_train.py) ============
def _make_divisible(v, divisor, min_value=None):
    if min_value is None:
        min_value = divisor
    new_v = max(min_value, int(v + divisor / 2) // divisor * divisor)
    if new_v < 0.9 * v:
        new_v += divisor
    return new_v


def hard_sigmoid(x, inplace=False):
    if inplace:
        return x.add_(3.).clamp_(0., 6.).div_(6.)
    return F.relu6(x + 3.) / 6.


class SqueezeExcite(nn.Module):
    def __init__(
        self, in_chs, se_ratio=0.25,
        reduced_base_chs=None, act_layer=nn.ReLU,
        gate_fn=hard_sigmoid, divisor=4, **_
    ):
        super().__init__()
        self.gate_fn = gate_fn
        reduced_chs = _make_divisible(
            (reduced_base_chs or in_chs) * se_ratio, divisor
        )
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv_reduce = nn.Conv2d(
            in_chs, reduced_chs, 1, bias=True
        )
        self.act1 = act_layer(inplace=True)
        self.conv_expand = nn.Conv2d(
            reduced_chs, in_chs, 1, bias=True
        )

    def forward(self, x):
        x_se = self.avg_pool(x)
        x_se = self.conv_reduce(x_se)
        x_se = self.act1(x_se)
        x_se = self.conv_expand(x_se)
        return x * self.gate_fn(x_se)


class ConvBnAct(nn.Module):
    def __init__(
        self, in_chs, out_chs, kernel_size,
        stride=1, act_layer=nn.ReLU
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_chs, out_chs, kernel_size, stride,
            kernel_size // 2, bias=False
        )
        self.bn1 = nn.BatchNorm2d(out_chs)
        self.act1 = act_layer(inplace=True)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn1(x)
        x = self.act1(x)
        return x


class GhostModuleV2(nn.Module):
    def __init__(
        self, inp, oup, kernel_size=1, ratio=2,
        dw_size=3, stride=1, relu=True,
        mode=None, args=None
    ):
        super().__init__()
        self.mode = mode
        self.oup = oup
        self.gate_fn = nn.Sigmoid()

        init_channels = math.ceil(oup / ratio)
        new_channels = init_channels * (ratio - 1)

        self.primary_conv = nn.Sequential(
            nn.Conv2d(
                inp, init_channels, kernel_size,
                stride, kernel_size // 2, bias=False
            ),
            nn.BatchNorm2d(init_channels),
            nn.ReLU(inplace=True) if relu else nn.Sequential(),
        )

        self.cheap_operation = nn.Sequential(
            nn.Conv2d(
                init_channels, new_channels, dw_size,
                1, dw_size // 2,
                groups=init_channels, bias=False
            ),
            nn.BatchNorm2d(new_channels),
            nn.ReLU(inplace=True) if relu else nn.Sequential(),
        )

        if self.mode == "attn":
            self.short_conv = nn.Sequential(
                nn.Conv2d(
                    inp, oup, kernel_size,
                    stride, kernel_size // 2, bias=False
                ),
                nn.BatchNorm2d(oup),
                nn.Conv2d(
                    oup, oup, kernel_size=(1, 5),
                    stride=1, padding=(0, 2),
                    groups=oup, bias=False
                ),
                nn.BatchNorm2d(oup),
                nn.Conv2d(
                    oup, oup, kernel_size=(5, 1),
                    stride=1, padding=(2, 0),
                    groups=oup, bias=False
                ),
                nn.BatchNorm2d(oup),
            )

    def forward(self, x):
        if self.mode == "original":
            x1 = self.primary_conv(x)
            x2 = self.cheap_operation(x1)
            out = torch.cat([x1, x2], dim=1)
            return out[:, :self.oup, :, :]

        if self.mode == "attn":
            res = self.short_conv(
                F.avg_pool2d(x, kernel_size=2, stride=2)
            )
            x1 = self.primary_conv(x)
            x2 = self.cheap_operation(x1)
            out = torch.cat([x1, x2], dim=1)
            gate = F.interpolate(
                self.gate_fn(res),
                size=(out.shape[-2], out.shape[-1]),
                mode="nearest"
            )
            return out[:, :self.oup, :, :] * gate

        raise ValueError("GhostModuleV2 mode must be 'original' or 'attn'")


class GhostBottleneckV2(nn.Module):
    def __init__(
        self, in_chs, mid_chs, out_chs,
        dw_kernel_size=3, stride=1,
        act_layer=nn.ReLU, se_ratio=0.,
        layer_id=None, args=None
    ):
        super().__init__()

        has_se = se_ratio is not None and se_ratio > 0
        self.stride = stride

        if layer_id is not None and layer_id <= 1:
            mode = "original"
        else:
            mode = "attn"

        self.ghost1 = GhostModuleV2(
            in_chs, mid_chs, relu=True,
            mode=mode, args=args
        )

        if self.stride > 1:
            self.conv_dw = nn.Conv2d(
                mid_chs, mid_chs, dw_kernel_size,
                stride=stride,
                padding=(dw_kernel_size - 1) // 2,
                groups=mid_chs, bias=False
            )
            self.bn_dw = nn.BatchNorm2d(mid_chs)

        self.se = (
            SqueezeExcite(mid_chs, se_ratio=se_ratio)
            if has_se else None
        )

        self.ghost2 = GhostModuleV2(
            mid_chs, out_chs, relu=False,
            mode="original", args=args
        )

        if in_chs == out_chs and self.stride == 1:
            self.shortcut = nn.Sequential()
        else:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_chs, in_chs, dw_kernel_size,
                    stride=stride,
                    padding=(dw_kernel_size - 1) // 2,
                    groups=in_chs, bias=False
                ),
                nn.BatchNorm2d(in_chs),
                nn.Conv2d(
                    in_chs, out_chs, 1,
                    stride=1, padding=0, bias=False
                ),
                nn.BatchNorm2d(out_chs),
            )

    def forward(self, x):
        residual = x

        x = self.ghost1(x)

        if self.stride > 1:
            x = self.conv_dw(x)
            x = self.bn_dw(x)

        if self.se is not None:
            x = self.se(x)

        x = self.ghost2(x)
        x = x + self.shortcut(residual)

        return x


class GhostNetV2(nn.Module):
    def __init__(
        self, cfgs, num_classes=1000,
        width=1.0, dropout=0.2,
        block=GhostBottleneckV2, args=None
    ):
        super().__init__()

        self.cfgs = cfgs
        self.dropout = dropout

        output_channel = _make_divisible(16 * width, 4)

        self.conv_stem = nn.Conv2d(
            3, output_channel, 3, 2, 1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(output_channel)
        self.act1 = nn.ReLU(inplace=True)

        input_channel = output_channel
        stages = []
        layer_id = 0

        for cfg in self.cfgs:
            layers = []

            for k, exp_size, c, se_ratio, s in cfg:
                output_channel = _make_divisible(c * width, 4)
                hidden_channel = _make_divisible(exp_size * width, 4)

                layers.append(
                    block(
                        input_channel,
                        hidden_channel,
                        output_channel,
                        k,
                        s,
                        se_ratio=se_ratio,
                        layer_id=layer_id,
                        args=args
                    )
                )

                input_channel = output_channel
                layer_id += 1

            stages.append(nn.Sequential(*layers))

        output_channel = _make_divisible(
            self.cfgs[-1][-1][2] * width, 4
        )

        stages.append(
            nn.Sequential(
                ConvBnAct(
                    input_channel,
                    output_channel,
                    1
                )
            )
        )

        input_channel = output_channel
        self.blocks = nn.Sequential(*stages)

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.conv_head = nn.Conv2d(
            input_channel, 1280, 1, 1, 0, bias=True
        )
        self.act2 = nn.ReLU(inplace=True)

        self.classifier = nn.Linear(
            1280, num_classes
        )

    def forward(self, x):
        x = self.conv_stem(x)
        x = self.bn1(x)
        x = self.act1(x)

        x = self.blocks(x)

        x = self.global_pool(x)
        x = self.conv_head(x)
        x = self.act2(x)

        x = x.view(x.size(0), -1)

        if self.dropout > 0:
            x = F.dropout(
                x,
                p=self.dropout,
                training=self.training
            )

        return self.classifier(x)


def create_ghostnetv2(
    num_classes,
    width=1.0,
    dropout=0.2,
    args=None
):
    cfgs = [
        [[3, 16, 16, 0, 1]],
        [[3, 48, 24, 0, 2]],
        [[3, 72, 24, 0, 1]],
        [[5, 72, 40, 0.25, 2]],
        [[5, 120, 40, 0.25, 1]],
        [[3, 240, 80, 0, 2]],
        [
            [3, 200, 80, 0, 1],
            [3, 184, 80, 0, 1],
            [3, 184, 80, 0, 1],
            [3, 480, 112, 0.25, 1],
            [3, 672, 112, 0.25, 1],
        ],
        [[5, 672, 160, 0.25, 2]],
        [
            [5, 960, 160, 0, 1],
            [5, 960, 160, 0.25, 1],
            [5, 960, 160, 0, 1],
            [5, 960, 160, 0.25, 1],
        ],
    ]

    return GhostNetV2(
        cfgs,
        num_classes=num_classes,
        width=width,
        dropout=dropout,
        args=args
    )
# ======================================================================================================

class GhostNetV2Classifier(torch.nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.backbone = create_ghostnetv2(num_classes=num_classes, width=WIDTH, dropout=DROPOUT)
    def forward(self, x): return self.backbone(x)

model = GhostNetV2Classifier(len(CLASSES)).to(device)
print(f"{MODEL_NAME} | pretrained={PRETRAINED} | params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")
if not PRETRAINED: print("Note: GhostNetV2 is trained from random weights (no pretrained checkpoint in the script).")


ghostnetv2 | pretrained=True | params: 3.7M


In [7]:
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_recall_fscore_support, confusion_matrix)
import torch.nn.functional as F

def compute_metrics(y_true, y_pred, loss):
    mp, mr, mf, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=range(len(CLASSES)), zero_division=0)
    return dict(loss=loss, acc=accuracy_score(y_true, y_pred), bal_acc=balanced_accuracy_score(y_true, y_pred),
                macro_p=mp, macro_r=mr, macro_f1=mf,
                per_class=pd.DataFrame({"precision": p, "recall": r, "f1": f, "support": s}, index=CLASSES),
                cm=confusion_matrix(y_true, y_pred, labels=range(len(CLASSES))))

def print_metrics(name, m, plot_cm=False):
    print(f"  [{name}] loss={m['loss']:.4f} | acc={m['acc']:.4f} | balanced_acc={m['bal_acc']:.4f} | "
          f"macro_P={m['macro_p']:.4f} | macro_R={m['macro_r']:.4f} | macro_F1={m['macro_f1']:.4f}")
    print(m["per_class"].round(4).to_string())
    print("  Confusion matrix (rows=true, cols=pred):")
    print(pd.DataFrame(m["cm"], index=[f"T_{c}" for c in CLASSES], columns=[f"P_{c}" for c in CLASSES]).to_string())
    if plot_cm:
        fig, ax = plt.subplots(figsize=(5, 4.2)); ax.imshow(m["cm"], cmap="Blues")
        ax.set_xticks(range(5)); ax.set_xticklabels(CLASSES, rotation=45); ax.set_yticks(range(5)); ax.set_yticklabels(CLASSES)
        for i in range(5):
            for j in range(5): ax.text(j, i, m["cm"][i, j], ha="center", va="center", color="red")
        ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"{name} confusion matrix"); plt.tight_layout(); plt.show()

class GhostNetGradCAM:
    """Grad-CAM on the last feature stage of GhostNetV2 (ConvBnAct, 960 x 7 x 7, just before global pooling)."""
    def __init__(self, model):
        self.model = model
        self.layer = model.backbone.blocks[-1]
    def __call__(self, x, class_idx=None):
        store = {}
        def fwd(m, i, o):
            store["a"] = o
            o.register_hook(lambda g: store.__setitem__("g", g))
        h = self.layer.register_forward_hook(fwd)
        self.model.zero_grad(set_to_none=True)
        with torch.enable_grad():
            x = x.clone().requires_grad_(True)
            out = self.model(x)
            if class_idx is None: class_idx = out.argmax(1)
            out.gather(1, class_idx[:, None]).sum().backward()
        h.remove()
        a, g = store["a"], store["g"]                       # (B, C, H, W)
        w = g.mean((2, 3), keepdim=True)                    # channel weights
        cam = (w * a).sum(1, keepdim=True).relu()           # (B, 1, H, W)
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
        mn, mx = cam.amin((1, 2), keepdim=True), cam.amax((1, 2), keepdim=True)
        return ((cam - mn) / (mx - mn + 1e-8)).detach().cpu().numpy(), out.argmax(1).detach().cpu().numpy()

gradcam = GhostNetGradCAM(model)
_mean, _std = torch.tensor(MEAN).view(3, 1, 1), torch.tensor(STD).view(3, 1, 1)

def show_gradcam(df, n_per_class=2, title="Grad-CAM", save=None):
    model.eval()
    ds = LesionDS(df, eval_tf); rows = []
    for c in range(len(CLASSES)):
        idx = df.index[df["label"] == c].tolist()[:n_per_class]
        rows.append(idx)
    fig, axes = plt.subplots(2 * n_per_class, len(CLASSES), figsize=(3 * len(CLASSES), 6 * n_per_class / 1.0 * 0.5 * 2 / 2 + 1))
    axes = np.atleast_2d(axes)
    for c, idxs in enumerate(rows):
        for k, i in enumerate(idxs):
            img, y = ds[i]
            cam, pred = gradcam(img[None].to(device))
            rgb = (img * _std + _mean).clamp(0, 1).permute(1, 2, 0).numpy()
            heat = plt.cm.jet(cam[0])[..., :3]
            axes[2 * k, c].imshow(rgb); axes[2 * k + 1, c].imshow(0.55 * rgb + 0.45 * heat)
            axes[2 * k, c].set_title(f"true: {CLASSES[y]}", fontsize=9)
            axes[2 * k + 1, c].set_title(f"pred: {CLASSES[pred[0]]}", fontsize=9,
                                         color="green" if pred[0] == y else "red")
        for r in range(2 * n_per_class): axes[r, c].axis("off")
    fig.suptitle(title); plt.tight_layout()
    if save: plt.savefig(save, dpi=110, bbox_inches="tight")
    plt.show()

In [12]:
import time,math,os
from tqdm.auto import tqdm
from torch.amp import autocast,GradScaler

counts=train_df["label"].value_counts().reindex(range(len(CLASSES))).values.astype(float)
cls_w=torch.tensor((counts.sum()/(len(CLASSES)*counts))**0.5,dtype=torch.float32).to(device)
cls_w=cls_w/cls_w.mean()
print("class weights:",dict(zip(CLASSES,cls_w.cpu().numpy().round(3))))
criterion=torch.nn.CrossEntropyLoss(weight=cls_w,label_smoothing=0.05)
eval_criterion=torch.nn.CrossEntropyLoss(weight=cls_w)

head_p=list(model.backbone.classifier.parameters())
head_ids={id(p) for p in head_p}
body_p=[p for p in model.parameters() if id(p) not in head_ids]
opt=torch.optim.AdamW([{"params":body_p,"lr":LR},{"params":head_p,"lr":LR*10}],weight_decay=WEIGHT_DECAY)

total_steps,warm=EPOCHS*len(train_loader),len(train_loader)
sched=torch.optim.lr_scheduler.LambdaLR(opt,lambda s:(s+1)/warm if s<warm else 0.5*(1+math.cos(math.pi*(s-warm)/max(1,total_steps-warm))))
scaler=GradScaler(enabled=device.type=="cuda")

CHECKPOINT_PATH="/content/checkpoint_ghostnetv2.pth"
BEST_MODEL_PATH="/content/best_ghostnetv2.pth"

start_epoch=1
best_f1,bad,history=-1,0,[]

if os.path.exists(CHECKPOINT_PATH):
    print(f"Found checkpoint at {CHECKPOINT_PATH}. Resuming training...")
    checkpoint=torch.load(CHECKPOINT_PATH,map_location=device,weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    opt.load_state_dict(checkpoint["optimizer_state_dict"])
    sched.load_state_dict(checkpoint["scheduler_state_dict"])
    scaler.load_state_dict(checkpoint["scaler_state_dict"])
    start_epoch=checkpoint["epoch"]+1
    best_f1=checkpoint["best_f1"]
    bad=checkpoint["bad"]
    history=checkpoint["history"]
    print(f"Resuming from Epoch {start_epoch} (Best val macro-F1 so far: {best_f1:.4f})")

@torch.no_grad()
def evaluate(loader):
    model.eval()
    ys,ps,tot=[],[],0.0
    for x,y in loader:
        x,y=x.to(device,non_blocking=True),y.to(device,non_blocking=True)
        with autocast(device.type,enabled=device.type=="cuda"):
            out=model(x)
        tot+=eval_criterion(out.float(),y).item()*len(y)
        ys+=y.cpu().tolist()
        ps+=out.argmax(1).cpu().tolist()
    return compute_metrics(ys,ps,tot/len(loader.dataset))

val_cam_df=val_df.groupby("label",group_keys=False).head(2).reset_index(drop=True)

for epoch in range(start_epoch,EPOCHS+1):
    t0=time.time()
    model.train()
    ys,ps,tot=[],[],0.0

    print("="*70)
    print(f"Epoch {epoch}/{EPOCHS}")
    print("="*70)

    for x,y in tqdm(train_loader,desc=f"Training Epoch {epoch}/{EPOCHS}",unit="batch"):
        x,y=x.to(device,non_blocking=True),y.to(device,non_blocking=True)

        opt.zero_grad(set_to_none=True)

        with autocast(device.type,enabled=device.type=="cuda"):
            out=model(x)
            loss=criterion(out.float(),y)

        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()

        tot+=loss.item()*len(y)
        ys+=y.cpu().tolist()
        ps+=out.argmax(1).cpu().tolist()

    tr=compute_metrics(ys,ps,tot/len(train_loader.dataset))
    va=evaluate(val_loader)

    train_wp,train_wr,train_wf,_=precision_recall_fscore_support(ys,ps,average="weighted",zero_division=0)

    val_ys,val_ps=[],[]
    model.eval()
    with torch.no_grad():
        for x,y in val_loader:
            x,y=x.to(device,non_blocking=True),y.to(device,non_blocking=True)
            with autocast(device.type,enabled=device.type=="cuda"):
                out=model(x)
            val_ys+=y.cpu().tolist()
            val_ps+=out.argmax(1).cpu().tolist()

    val_wp,val_wr,val_wf,_=precision_recall_fscore_support(val_ys,val_ps,average="weighted",zero_division=0)

    print("\nTRAIN METRICS")
    print("-"*70)
    print(f"Train Loss              : {tr['loss']:.4f}")
    print(f"Train Accuracy          : {tr['acc']:.4f}")
    print(f"Train Balanced Accuracy : {tr['bal_acc']:.4f}")
    print(f"Train Macro Precision   : {tr['macro_p']:.4f}")
    print(f"Train Macro Recall      : {tr['macro_r']:.4f}")
    print(f"Train Macro F1          : {tr['macro_f1']:.4f}")
    print(f"Train Weighted Precision: {train_wp:.4f}")
    print(f"Train Weighted Recall   : {train_wr:.4f}")
    print(f"Train Weighted F1       : {train_wf:.4f}")

    for c in range(len(CLASSES)):
        row=tr["per_class"].iloc[c]
        print(f"{CLASSES[c]:15s} | Precision: {row['precision']:.4f} | Recall: {row['recall']:.4f} | F1: {row['f1']:.4f} | Support: {int(row['support'])}")

    print("\nVALIDATION METRICS")
    print("-"*70)
    print(f"Validation Loss              : {va['loss']:.4f}")
    print(f"Validation Accuracy          : {va['acc']:.4f}")
    print(f"Validation Balanced Accuracy : {va['bal_acc']:.4f}")
    print(f"Validation Macro Precision   : {va['macro_p']:.4f}")
    print(f"Validation Macro Recall      : {va['macro_r']:.4f}")
    print(f"Validation Macro F1          : {va['macro_f1']:.4f}")
    print(f"Validation Weighted Precision: {val_wp:.4f}")
    print(f"Validation Weighted Recall   : {val_wr:.4f}")
    print(f"Validation Weighted F1       : {val_wf:.4f}")

    for c in range(len(CLASSES)):
        row=va["per_class"].iloc[c]
        print(f"{CLASSES[c]:15s} | Precision: {row['precision']:.4f} | Recall: {row['recall']:.4f} | F1: {row['f1']:.4f} | Support: {int(row['support'])}")

    print("\nTRAIN CONFUSION MATRIX")
    print("-"*70)
    print(pd.DataFrame(tr["cm"],index=[f"T_{c}" for c in CLASSES],columns=[f"P_{c}" for c in CLASSES]).to_string())

    print("\nVALIDATION CONFUSION MATRIX")
    print("-"*70)
    print(pd.DataFrame(va["cm"],index=[f"T_{c}" for c in CLASSES],columns=[f"P_{c}" for c in CLASSES]).to_string())

    print("\nGRAD-CAM")
    print("-"*70)
    show_gradcam(val_cam_df,2,title=f"Grad-CAM (val) - epoch {epoch}")

    history.append({
        "epoch":epoch,
        "train_loss":tr["loss"],
        "train_acc":tr["acc"],
        "train_bal_acc":tr["bal_acc"],
        "train_macro_p":tr["macro_p"],
        "train_macro_r":tr["macro_r"],
        "train_macro_f1":tr["macro_f1"],
        "train_weighted_p":train_wp,
        "train_weighted_r":train_wr,
        "train_weighted_f1":train_wf,
        "val_loss":va["loss"],
        "val_acc":va["acc"],
        "val_bal_acc":va["bal_acc"],
        "val_macro_p":va["macro_p"],
        "val_macro_r":va["macro_r"],
        "val_macro_f1":va["macro_f1"],
        "val_weighted_p":val_wp,
        "val_weighted_r":val_wr,
        "val_weighted_f1":val_wf
    })

    if va["macro_f1"]>best_f1:
        best_f1,bad=va["macro_f1"],0
        torch.save(model.state_dict(),BEST_MODEL_PATH)
        print("\nNEW BEST MODEL SAVED")
        print(f"Best Validation Macro-F1 : {best_f1:.4f}")
    else:
        bad+=1

    torch.save({
        "epoch":epoch,
        "model_state_dict":model.state_dict(),
        "optimizer_state_dict":opt.state_dict(),
        "scheduler_state_dict":sched.state_dict(),
        "scaler_state_dict":scaler.state_dict(),
        "best_f1":best_f1,
        "bad":bad,
        "history":history
    },CHECKPOINT_PATH)

    print(f"\nEpoch time: {(time.time()-t0)/60:.2f} minutes")
    print(f"Checkpoint saved: {CHECKPOINT_PATH}")

class weights: {'melanoma': np.float32(1.234), 'nevus': np.float32(0.982), 'bcc': np.float32(0.524), 'scc': np.float32(1.177), 'keratoses': np.float32(1.083)}
Found checkpoint at /content/checkpoint_ghostnetv2.pth. Resuming training...
Resuming from Epoch 3 (Best val macro-F1 so far: 0.3090)
Epoch 3/100


Training Epoch 3/100:   0%|          | 0/480 [00:00<?, ?batch/s]

KeyboardInterrupt: 

In [ ]:
model.load_state_dict(torch.load("/content/best_ghostnetv2.pth", map_location=device))
te = evaluate(test_loader)
print("#" * 40, "TEST RESULTS (best checkpoint)", "#" * 40)
print_metrics("TEST", te, plot_cm=True)
test_cam_df = test_df.groupby("label", group_keys=False).head(4).reset_index(drop=True)
show_gradcam(test_cam_df, 4, title="Grad-CAM (test, best model)", save="/content/gradcam_test.png")

h = pd.DataFrame(history)
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
for a, (k, t) in zip(ax, [("loss", "Loss"), ("acc", "Accuracy"), ("macro_f1", "Macro F1")]):
    a.plot(h["epoch"], h[f"train_{k}"], label="train"); a.plot(h["epoch"], h[f"val_{k}"], label="val"); a.set_title(t); a.legend()
plt.show()
# Download the weights if you want them:  from google.colab import files; files.download("/content/best_ghostnetv2.pth")